In [1]:
import polars as pl
import numpy as np
import pandas as pd
import os 

from pathlib import Path
from datetime import date
from dotenv import load_dotenv
from typing import Callable

from ohlc_dss_model.data import (
    load_parquet, remove_incomplete_days, write_parquet,
    intraday_session_tagging, session_tagging, 
    filter_valid_sessions
)

from ohlc_dss_model.features import(
    aggregate_sessions, yang_zhang,
    PRE_NY_SPEC, FULL_DAY_SPEC,
    calculate_excursion_bands, assign_direction,
    detect_pivots, pivot_extraction, build_pivot_features,
    build_event_table, encode_news_context, inspect_event_table,
    get_calendar_index, build_fred_macro, build_individual_event_flags
)

from ohlc_dss_model.utils import (
    convert_to_timezone, plot_session
)

from ohlc_dss_model.config import config

In [2]:
def load_raw_data(file_path: Path) -> pl.DataFrame:
    raw_data = load_parquet(file_path)
    raw_data = convert_to_timezone(raw_data)
    raw_data = session_tagging(raw_data)
    raw_data = intraday_session_tagging(raw_data)
    raw_data = remove_incomplete_days(raw_data)
    return raw_data.select(pl.col([
        "DateTime", "Session", "Intraday_Session", 
        "Open", "High", "Low", "Close", "Volume"
    ]))

In [3]:
def load_aggregated_data(df: pl.DataFrame) -> pl.DataFrame:
    aggregated_data = aggregate_sessions(df)
    aggregated_data = filter_valid_sessions(aggregated_data)

    aggregated_data = aggregated_data.with_columns(
        pl.col("O_Pre_Target_1").alias("O_Ref")
    )

    aggregated_data = yang_zhang(aggregated_data, FULL_DAY_SPEC, mode="historical")
    aggregated_data = yang_zhang(aggregated_data, PRE_NY_SPEC, mode="today")

    aggregated_data = assign_direction(aggregated_data)
    aggregated_data = calculate_excursion_bands(aggregated_data)
    return aggregated_data

In [4]:
def _incremental_loader(
    folder_path: Path,
    file_name: str,
    df: pl.DataFrame,
    build_fn,
) -> pl.DataFrame:
    file_path = folder_path / f"{file_name}.parquet"

    min_current = df.select(pl.col("Session").min()).item()
    max_current = df.select(pl.col("Session").max()).item()

    if file_path.exists():
        print(f"[{file_name}] Loading cached table...")
        table = load_parquet(file_path)

        max_session = table.select(pl.col("Session").max()).item()
        print(f"[{file_name}] Cached max session: {max_session}")

        if max_session < max_current:
            print(f"[{file_name}] Extending to {max_current}...")
            extended = build_fn(max_session + 1, max_current)

            if not extended.is_empty():
                new_sessions = extended.select(pl.col("Session")).unique()
                table = table.join(new_sessions, on="Session", how="anti")
                table = pl.concat([table, extended])

            write_parquet(table, file_name, folder_path)

    else:
        print(f"[{file_name}] No cache found. Building fresh...")
        table = build_fn(min_current, max_current)
        write_parquet(table, file_name, folder_path)

    print(f"[{file_name}] Ready.")
    return table

In [5]:
def _load_event_table(folder_path, df, api_key):
    return _incremental_loader(
        folder_path,
        "event_table",
        df,
        build_fn=lambda start, end: build_event_table(start, end, api_key=api_key),
    )


def _load_fred_macro(folder_path, df, api_key):
    return _incremental_loader(
        folder_path,
        "fred_macro_table",
        df,
        build_fn=lambda start, end: build_fred_macro(
            df,
            api_key=api_key,
            start_date=start,
            end_date=end,
        ),
    )


def _load_individual_event_flags(folder_path, df, api_key):
    return _incremental_loader(
        folder_path,
        "individual_event_flags",
        df,
        build_fn=lambda start, end: build_individual_event_flags(
            df,
            api_key=api_key,
            start=start,
            end=end,
        ),
    )

In [6]:
def get_macro_features(folder_path: Path, df: pl.DataFrame, api_key: str) -> pl.DataFrame:
    event_table = _load_event_table(folder_path, df, api_key)

    if event_table.is_empty():
        print("[get_macro_features] No events found.")
        return pl.DataFrame()

    print("[get_macro_features] Encoding news context...")
    macro_features = encode_news_context(df, event_table)

    print("[get_macro_features] Adding calendar features...")
    macro_features = get_calendar_index(macro_features)

    print("[get_macro_features] Joining FRED macro...")
    fred_macro = _load_fred_macro(folder_path, df, api_key)
    macro_features = macro_features.join(fred_macro, on="Session", how="left")

    print("[get_macro_features] Joining individual event flags...")
    flags = _load_individual_event_flags(folder_path, df, api_key)
    macro_features = macro_features.join(flags, on="Session", how="left")

    return macro_features

In [7]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")

In [8]:
raw_data_30m = load_raw_data(config.data.raw_folder_path / "nq_30m.parquet")
raw_data_1m = load_raw_data(config.data.raw_folder_path / "nq_1m.parquet")

aggregated_data = load_aggregated_data(raw_data_30m)
print(aggregated_data.select(["Session"]).max().item())

aggregated_data = get_macro_features(config.data.processed_folder_path, aggregated_data, FRED_API_KEY)

2026-04-16
[event_table] Loading cached table...
[event_table] Cached max session: 2026-04-29
[event_table] Ready.
[get_macro_features] Encoding news context...
[get_macro_features] Adding calendar features...
[get_macro_features] Joining FRED macro...
[fred_macro_table] Loading cached table...
[fred_macro_table] Cached max session: 2026-04-16
[fred_macro_table] Ready.
[get_macro_features] Joining individual event flags...
[individual_event_flags] Loading cached table...
[individual_event_flags] Cached max session: 2026-04-16
[individual_event_flags] Ready.


In [9]:
aggregated_data.tail(5)

Session,O_Target_2,O_Pre_Target_1,O_Pre_Target_2,O_Target_1,H_Target_2,H_Pre_Target_1,H_Pre_Target_2,H_Target_1,L_Target_2,L_Pre_Target_1,L_Pre_Target_2,L_Target_1,C_Target_2,C_Pre_Target_1,C_Pre_Target_2,C_Target_1,O_Ref,Sigma_Historical,Sigma_Today,Z_Body,Z_Sigma,Tau,Direction,_delta_t,Band_AE_Pos_Center,Band_AE_Neg_Center,Band_FE_Pos_Center,Band_FE_Neg_Center,Band_AE_Neg_Upper,Band_AE_Neg_Lower,Band_AE_Pos_Upper,Band_AE_Pos_Lower,Band_FE_Neg_Upper,Band_FE_Neg_Lower,Band_FE_Pos_Upper,Band_FE_Pos_Lower,e_today,e_yesterday,e_tomorrow,day_of_week,month,week_of_month,vix_t1,us10y_t1,us2y_t1,effr_t1,10y_2y_spread_t1,vix_pct_rank_1y_t1,vix_5d_delta,us10y_5d_delta,is_fomc_day,is_fomc_week,days_to_fomc,is_nfp_day,is_cpi_day,is_core_cpi_day
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,i16,bool,bool,bool
2026-04-10,25265.5,25237.0,25229.25,25324.5,25338.0,25303.75,25386.0,25393.0,25223.25,25193.0,25202.75,25244.75,25333.0,25228.25,25336.25,25264.5,25237.0,0.015757,0.006078,0.240953,0.691766,0.480928,"""neutral""",53.69565,25448.86126,25025.13874,25716.320867,24757.679133,25078.83439,24971.44309,25502.55691,25395.16561,24811.374783,24703.983482,25770.016518,25662.625217,3,2,0,5,4,2,19.49,4.29,3.78,3.64,0.51,0.688022,-4.38,-0.02,false,false,19,false,true,true
2026-04-13,25411.0,24980.0,25115.5,25125.75,25599.25,25119.0,25194.75,25360.75,25396.0,24904.5,25073.5,25120.25,25585.5,25115.5,25124.75,25309.5,24980.0,0.017534,0.016756,1.365905,0.797482,0.447919,"""bullish""",39.361303,25142.484743,24817.515257,25266.801046,24693.198954,24856.87656,24778.153954,25181.846046,25103.12344,24732.560256,24653.837651,25306.162349,25227.439744,0,0,1,1,4,2,19.23,4.31,3.81,3.64,0.5,0.67688,-4.38,-0.04,false,false,16,false,false,false
2026-04-14,25914.25,25575.25,25586.0,25692.75,26003.25,25619.25,25701.0,25899.75,25884.25,25560.0,25581.5,25651.75,25990.0,25584.75,25694.5,25875.75,25575.25,0.018336,0.005116,0.877334,0.862704,0.430655,"""bullish""",44.844586,25765.852443,25384.647557,25932.110218,25218.389782,25429.492143,25339.802971,25810.697029,25721.007857,25263.234368,25173.545196,25976.954804,25887.265632,1,0,0,2,4,2,19.12,4.3,3.78,3.64,0.52,0.668524,-5.05,-0.04,false,false,15,false,false,false
2026-04-15,26144.25,25985.75,25988.5,25974.5,26377.5,26058.0,26048.0,26215.5,26136.75,25966.0,25933.75,25970.0,26359.5,25988.0,25975.0,26196.75,25985.75,0.018314,0.003498,0.779765,0.889093,0.424215,"""bullish""",47.64735,26128.518606,25842.981394,26361.341152,25610.158848,25890.628744,25795.334044,26176.165956,26080.871256,25657.806198,25562.511497,26408.988503,26313.693802,0,1,2,3,4,3,18.36,4.26,3.76,3.64,0.5,0.604457,-7.42,-0.07,false,false,14,false,false,false
2026-04-16,26474.25,26350.0,26479.75,26427.75,26509.75,26486.0,26488.0,26543.0,26355.0,26345.0,26364.5,26275.5,26464.25,26480.75,26425.5,26533.25,26350.0,0.017909,0.004608,0.241584,0.90692,0.420025,"""neutral""",48.25674,26487.952665,26212.047335,26742.43367,25957.56633,26260.304075,26163.790595,26536.209405,26439.695925,26005.823069,25909.30959,26790.69041,26694.176931,2,0,0,4,4,3,18.17,4.29,3.76,3.64,0.53,0.590529,-2.87,0.0,false,false,13,false,false,false
